## 메타퀘스트 릴레이 서버 실행

메타퀘스트 사용 시에만, 터미널을 열어 프로젝트 루트 디렉토리에서 다음 명령을 실행하세요.

```bash
YAM_MUJOCO_FRAME_PATH=/tmp/yam-mujoco-frame.jpg \
CAM_WIDTH=640 CAM_HEIGHT=480 CAM_FPS=15 \
.venv/bin/python -m yam_control.teleop.mujoco_relay \
  --host 0.0.0.0 \
  --port 8443 \
  --ssl-keyfile certs/key.pem \
  --ssl-certfile certs/cert.pem
```

서버가 실행되면 메타퀘스트 브라우저에서 `https://192.168.1.2:8443`(PC에 따라 주소는 달라질 수 있음)에 접속하세요.

## 1. 실행 설정

In [ ]:
import os
from pathlib import Path
import sys

from IPython.display import display

# Quest stream용 MuJoCo offscreen renderer에 EGL backend를 사용
os.environ.setdefault('MUJOCO_GL', 'egl')

PROJECT_ROOT: Path = Path.cwd().resolve()
YAM_ABC_ROOT: Path = PROJECT_ROOT / 'third_party' / 'yam-abc-reproduce'
I2RT_ROOT: Path = YAM_ABC_ROOT / 'third_party' / 'i2rt'
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(YAM_ABC_ROOT))
sys.path.insert(0, str(I2RT_ROOT))

from yam_control import RunConfig, RuntimeDependencies, build_run_config, create_session
from yam_control.config import CameraConfig, ExecutionTarget, QuestConfig, QuestControllerHand, QuestDisplayMode, RobotConfig, RunMode, TeleopSourceType, VLAType
from yam_control.session import RunSession
from yam_control.types import EpisodeState

MODE: RunMode = 'teleop'

TELEOP_SOURCE: TeleopSourceType = 'quest3'

VLA_TYPE: VLAType = 'pi0.5'

EXECUTION_TARGET: ExecutionTarget = 'mujoco'

TASK_PROMPT: str = 'pick up the object'

SAVE_TELEOP_DATA: bool = False

USE_SAFETY_GATE: bool = False

USE_RTC: bool = False

EPISODE_STEPS: int = 900

SIMULATION_EPISODES: int = 1

QUEST_TRANSLATION_SCALE: float = 0.5

QUEST_ROTATION_SCALE: float = 0.5

QUEST_POSITION_REACH_LIMIT_M: float = 0.10

QUEST_ROTATION_REACH_LIMIT_RAD: float = 0.35

QUEST_MAX_JOINT_DELTA_RAD: float = 0.06

QUEST_STREAM_WIDTH: int = 640

QUEST_STREAM_HEIGHT: int = 480

QUEST_STREAM_FPS: float = 15.0

QUEST_STREAM_JPEG_QUALITY: int = 70

CHECKPOINT_URI: str = ''

CHECKPOINT_REVISION: str | None = None

POLICY_CONFIG_NAME: str = ''

CONTROL_HZ: float = 30.0

FOLLOWER_CAN_CHANNEL: str = 'can0'

LEADER_CAN_CHANNEL: str = 'can1'

GRIPPER_TYPE: str = 'linear_4310'

EPISODE_INITIAL_POSE: tuple[float, ...] = (0.0, 1.2, 0.9, 0.0, 0.0, 0.0, 1.0)

HUMAN_RESET_POSE: tuple[float, ...] | None = None

QUEST_RELAY_HOST: str = '127.0.0.1'

QUEST_RELAY_PORT: int = 8443

QUEST_CONTROLLER_HAND: QuestControllerHand = 'right'

QUEST_DISPLAY_MODE: QuestDisplayMode = 'robot_camera'

QUEST_STREAM_FRAME_PATH: str = '/tmp/yam-mujoco-frame.jpg'

CAMERA_ROLES: tuple[str, ...] = ()

DATA_ROOT: str = 'data/episodes'

robot_config: RobotConfig = RobotConfig(
    follower_can_channel=FOLLOWER_CAN_CHANNEL,
    leader_can_channel=LEADER_CAN_CHANNEL,
    gripper_type=GRIPPER_TYPE,
    episode_initial_pose=EPISODE_INITIAL_POSE,
    human_reset_pose=HUMAN_RESET_POSE,
)
quest_config: QuestConfig = QuestConfig(
    relay_host=QUEST_RELAY_HOST,
    relay_port=QUEST_RELAY_PORT,
    display_mode=QUEST_DISPLAY_MODE,
    controller_hand=QUEST_CONTROLLER_HAND,
    translation_scale=QUEST_TRANSLATION_SCALE,
    rotation_scale=QUEST_ROTATION_SCALE,
    position_reach_limit_m=QUEST_POSITION_REACH_LIMIT_M,
    rotation_reach_limit_rad=QUEST_ROTATION_REACH_LIMIT_RAD,
    max_joint_delta_rad=QUEST_MAX_JOINT_DELTA_RAD,
    stream_frame_path=QUEST_STREAM_FRAME_PATH,
    stream_width=QUEST_STREAM_WIDTH,
    stream_height=QUEST_STREAM_HEIGHT,
    stream_fps=QUEST_STREAM_FPS,
    stream_jpeg_quality=QUEST_STREAM_JPEG_QUALITY,
)
camera_config: CameraConfig = CameraConfig(roles=CAMERA_ROLES)
config: RunConfig = build_run_config(
    mode=MODE,
    teleop_source=TELEOP_SOURCE,
    save_teleop_data=SAVE_TELEOP_DATA,
    vla_type=VLA_TYPE,
    checkpoint_uri=CHECKPOINT_URI,
    checkpoint_revision=CHECKPOINT_REVISION,
    policy_config_name=POLICY_CONFIG_NAME,
    use_rtc=USE_RTC,
    execution_target=EXECUTION_TARGET,
    use_safety_gate=USE_SAFETY_GATE,
    control_hz=CONTROL_HZ,
    task_prompt=TASK_PROMPT,
    data_root=DATA_ROOT,
    robot=robot_config,
    quest=quest_config,
    camera=camera_config,
)

## 2. Session 생성

In [ ]:
dependencies: RuntimeDependencies = RuntimeDependencies()
session: RunSession = create_session(
    config=config,
    dependencies=dependencies,
)

## 3. 메타퀘스트, MuJoCo 연결

메타퀘스트와 MuJoCo backend를 연결하고 첫 유효 `xr_frame`을 최대 10초 기다립니다.

In [ ]:
# 선택된 로봇과 action source를 연결
session.connect()

## 4. Episode 초기화

로봇을 `episode_initial_pose`로 보내고 에피소드 시작 상태를 준비합니다. MuJoCo environment는 자동으로 reset됩니다.

In [ ]:
# 실제 로봇은 human reset pose로 이동하고 MuJoCo는 environment를 자동 reset
session.prepare_episode()

## 5. Teleoperation 실행

에피소드를 시작합니다. (메타퀘스트 사용 시)리모컨의 Grip(중지로 누르는 버튼)을 누르는 동안만 로봇이 움직입니다.

In [ ]:
# 실제 로봇은 episode 하나를 실행하고 MuJoCo는 여러 에피소드를 연속 실행
episode_result: EpisodeState | tuple[EpisodeState, ...]
if config.common.execution_target == 'real':
    episode_result = session.run_prepared_episode(max_steps=EPISODE_STEPS)
else:
    episode_result = session.run_simulation_episodes(
        episode_count=SIMULATION_EPISODES,
        max_steps=EPISODE_STEPS,
    )
display(episode_result)
display(session.action_producer_diagnostics())
display({'control_loop': session.control_loop_diagnostics()})

## 6. Session 종료

메타퀘스트 controller reader, MuJoCo viewer와 session resource를 안전하게 해제합니다.

In [ ]:
# Controller, VLA와 robot resource를 해제
session.close()